# 02. 센서 신호 전처리

본 노트북은 JD의 다음 항목을 실행·검증한다.

- **결측치 보간** — 선형 / 큐빅 스플라인 / KNN
- **노이즈 필터링** — Butterworth 대역통과 / Wavelet denoising
- **이상치 탐지** — IQR / MAD / 이동창 Z-score / Isolation Forest
- **다중 센서 시간 동기화 및 리샘플링** — polyphase resample, merge_asof

대응 모듈: `src/career_kia/preprocessing/{imputation,filtering,outliers,synchronization}.py`

선행: `make data` (AI4I·CWRU 다운로드 + batch_master.parquet 생성)

In [ ]:
import sys
from pathlib import Path

# 레포 루트를 import 경로에 추가
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from career_kia.config import INTERIM_DIR, CWRU_FS_12K, TARGET_FS
from career_kia.data.loaders import load_cwru_signals, load_ai4i
from career_kia.preprocessing import imputation, filtering, outliers, synchronization

plt.rcParams['figure.figsize'] = (11, 3)
plt.rcParams['axes.grid'] = True

## 1. 공정 파라미터 결측치 보간

In [ ]:
batch = pd.read_parquet(INTERIM_DIR / 'batch_master.parquet').sort_values('timestamp').reset_index(drop=True)
proc_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

# 인공 결측 주입 (2%)
rng = np.random.default_rng(0)
df = batch[proc_cols].copy()
for c in proc_cols:
    df.loc[rng.random(len(df)) < 0.02, c] = np.nan

print('결측 요약:')
display(imputation.summarize_missing(df))

In [ ]:
# 3가지 보간 결과 비교 (Torque 일부 구간)
target = 'Torque [Nm]'
window = slice(500, 600)

orig = batch[target].iloc[window]
with_nan = df[target].iloc[window]
lin = imputation.impute_series(df[target], method='linear').iloc[window]
spl = imputation.impute_series(df[target], method='spline').iloc[window]
knn = imputation.impute_dataframe(df, method='knn')[target].iloc[window]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(orig.index, orig.values, 'k-', label='원본', alpha=0.4)
ax.plot(with_nan.index, with_nan.values, 'o', label='결측 주입 후', markersize=3)
ax.plot(lin.index, lin.values, '--', label='선형 보간')
ax.plot(spl.index, spl.values, ':', label='스플라인 보간')
ax.plot(knn.index, knn.values, '-.', label='KNN 보간')
ax.set_title(f'{target} 보간 비교')
ax.legend()
plt.show()

## 2. 진동 신호 필터링 (Butterworth + Wavelet)

In [ ]:
cwru = load_cwru_signals()
# IR 결함 신호 선택
ir_sig = next(r for r in cwru if r.fault_type == 'IR').signal
fs = CWRU_FS_12K

sig_bp = filtering.butterworth(ir_sig, fs=fs, cutoff=(500, 5000), filter_type='bandpass', order=4)
sig_wd = filtering.wavelet_denoise(ir_sig, wavelet='db4', level=4)

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
t = np.arange(len(ir_sig)) / fs
axes[0].plot(t, ir_sig, lw=0.5); axes[0].set_title('원본 진동 (IR 결함)')
axes[1].plot(t, sig_bp, lw=0.5); axes[1].set_title('Butterworth 대역통과 500-5000Hz')
axes[2].plot(t, sig_wd, lw=0.5); axes[2].set_title('Wavelet denoise (db4, level 4)')
axes[-1].set_xlabel('시간 (초)')
plt.tight_layout()
plt.show()

## 3. 이상치 탐지

In [ ]:
torque = batch['Torque [Nm]'].copy()
# 가상의 스파이크 주입
torque.iloc[[1234, 5678, 9000]] = [200, -50, 300]

iqr = outliers.iqr_mask(torque, k=1.5)
mad = outliers.mad_mask(torque, threshold=3.5)
roll = outliers.rolling_zscore_mask(torque, window=100, threshold=3.5)
iso = outliers.isolation_forest_mask(batch[proc_cols].assign(Torque_spike=torque.values)[['Torque_spike'] + [c for c in proc_cols if c != 'Torque [Nm]']], contamination=0.005)

print(f'IQR 마스킹:        {iqr.sum():4d}')
print(f'MAD 마스킹:        {mad.sum():4d}')
print(f'이동창 Z-score:    {roll.sum():4d}')
print(f'IsolationForest:   {iso.sum():4d}')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(torque.values, lw=0.5, alpha=0.5, label='Torque')
ax.scatter(np.flatnonzero(iqr), torque[iqr], c='r', s=10, label='IQR flag')
ax.set_title('이상치 탐지 — IQR 시각화')
ax.legend()
plt.show()

## 4. 다중 센서 시간 동기화 및 리샘플링

진동(12 kHz)과 공정 파라미터(배치당 1샘플)는 타임베이스가 다르다. 본 예시는 진동 신호의 윈도우별 통계를 배치 주기(10분)로 집계하여 공정 파라미터와 `merge_asof`로 조인하는 흐름을 보여준다.

In [ ]:
# 두 주파수 리샘플 예시
orig_48k = np.tile(ir_sig, 4)[:48_000]
rs_12k = synchronization.resample_signal(orig_48k, fs_in=48_000, fs_out=12_000)
print(f'48kHz 원본: {len(orig_48k)} → 12kHz 리샘플: {len(rs_12k)}')

# 비동기 MES 이벤트 조인 예시
mes = pd.DataFrame({
    'timestamp': pd.to_datetime(['2026-01-01 00:03', '2026-01-01 00:17', '2026-01-01 00:55']),
    'event': ['툴교체', '이상알람', '점검']
})
sensor = batch[['batch_id', 'timestamp']].head(6)
joined = synchronization.merge_asof_aligned(sensor, mes, tolerance='20min')
display(joined)

## 요약

- 결측 보간: 선형/스플라인/KNN 3가지 비교 — 변수별 특성에 따라 선택 가능
- 필터링: Butterworth로 타겟 대역 추출, Wavelet으로 충격성 결함 보존하며 잡음 제거
- 이상치: 단변량(IQR/MAD) + 시계열(rolling z) + 다변량(IsolationForest) 조합
- 동기화: polyphase 리샘플 + merge_asof로 MES 이벤트 조인

다음 노트북(`03_윈도잉_피처엔지니어링.ipynb`)에서 이 전처리 결과 위에 윈도잉 기반 시간/주파수 도메인 피처를 생성한다.